#INIT

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *


# Read Customer

In [0]:
customers_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/bronze/sql/dbo.customers.csv"

df_customers = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customers_path)
)

display(df_customers)

# Check the data

# check nulls

In [0]:
print("Bronze customer count:", df_customers.count())

df_customers.printSchema()

display(
    df_customers.select(
        [
            count(when(col(c).isNull(), c)).alias(c)
            for c in df_customers.columns
        ]
    )
)

# Clean customers

In [0]:
df_customers_clean = (
    df_customers
    .dropDuplicates()
    .dropDuplicates(["customer_id"])
    .filter(col("customer_id").isNotNull())
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("phone", trim(col("phone")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn(
        "registration_date",
        to_date(col("registration_date"))
    )
)

# Validate

In [0]:
print("Bronze count :", df_customers.count())
print("Silver count :", df_customers_clean.count())

display(df_customers_clean)

# check duplicate ids
df_customers_clean.groupBy("customer_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

# Write as Delta to Silver


In [0]:
customers_silver_path = "abfss://customer360@stcustomers360dev01.dfs.core.windows.net/silver/customers/"

(
    df_customers_clean.write
    .format("delta")
    .mode("overwrite")
    .save(customers_silver_path)
)

# Verify Delta

In [0]:
df_customers_silver = (
    spark.read
    .format("delta")
    .load(customers_silver_path)
)

display(df_customers_silver)

print("Silver customer count:", df_customers_silver.count())